# CH101 free AI 3D autobuild

This notebook creates non-production CH101 review candidates. It uses the Tripo free trial when `TRIPO_API_KEY` is available, then supports Stable Fast 3D or TripoSR as single-view Colab fallbacks. It never enables Unity input or approves Gate B.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

CHARACTER_CODE = 'CH101'
PROVIDER = os.environ.get('RE_CAMP_AI3D_PROVIDER', 'tripo').lower()
CANDIDATE_COUNT = int(os.environ.get('RE_CAMP_AI3D_CANDIDATES', '4'))
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
TOOLS_COMMIT = os.environ.get('RE_CAMP_BLENDER_TOOLS_COMMIT', '6f78cd8')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
CONTENT_ROOT = Path('/content')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidates' / PROVIDER
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
REVIEW_DIR = OUTPUT_ROOT / 'review'
assert PROVIDER in {'tripo', 'sf3d', 'triposr'}
print({'provider': PROVIDER, 'candidateCount': CANDIDATE_COUNT, 'output': str(OUTPUT_ROOT)})


In [ ]:
def run(command, **kwargs):
    print('RUN:', ' '.join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
if shutil.which('blender') is None or shutil.which('xvfb-run') is None:
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'])
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
if TOOLS_COMMIT:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_COMMIT])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', TOOLS_COMMIT])
if not (ART_DIR / '.git').is_dir():
    run(['git', 'clone', ART_REPO_URL, ART_DIR])
run(['git', '-C', ART_DIR, 'fetch', 'origin', ART_COMMIT])
run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('blender:', shutil.which('blender'))


In [ ]:
prepare_script = TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py'
run([
    sys.executable, prepare_script,
    '--art-root', ART_DIR,
    '--output-dir', REFERENCE_DIR,
])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
reference_manifest = json.loads(REFERENCE_MANIFEST.read_text(encoding='utf-8'))
assert reference_manifest['artCommit'] == ART_COMMIT
assert reference_manifest['unityInputAllowed'] is False
print(json.dumps(reference_manifest, indent=2, ensure_ascii=False))


In [ ]:
provider_environment = os.environ.copy()
if PROVIDER == 'tripo':
    api_key = os.environ.get('TRIPO_API_KEY', '')
    try:
        from google.colab import userdata
        api_key = api_key or userdata.get('TRIPO_API_KEY')
    except Exception:
        pass
    command = [
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'tripo_api.py',
        '--reference-manifest', REFERENCE_MANIFEST,
        '--output-dir', CANDIDATE_DIR,
        '--candidate-count', str(CANDIDATE_COUNT),
    ]
    if api_key:
        provider_environment['TRIPO_API_KEY'] = api_key
        command.append('--execute')
    else:
        print('TRIPO_API_KEY is absent: creating a zero-credit dry-run plan only.')
    run(command, env=provider_environment)
else:
    contract = json.loads((TOOLS_DIR / 'contracts' / 'ch101_ai3d_free_pipeline_v001.json').read_text(encoding='utf-8'))
    provider_key = 'stableFast3D' if PROVIDER == 'sf3d' else 'tripoSR'
    provider_config = contract['providers'][provider_key]
    provider_repo = CONTENT_ROOT / f"provider-{PROVIDER}"
    if not (provider_repo / '.git').is_dir():
        run(['git', 'clone', provider_config['repository'], provider_repo])
    run(['git', '-C', provider_repo, 'fetch', 'origin', provider_config['commit']])
    run(['git', '-C', provider_repo, 'checkout', '--detach', provider_config['commit']])
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', provider_repo / 'requirements.txt'])
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            provider_environment['HF_TOKEN'] = hf_token
    except Exception:
        pass
    run([
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_open_source_provider.py',
        '--provider', PROVIDER,
        '--provider-repo', provider_repo,
        '--reference-manifest', REFERENCE_MANIFEST,
        '--output-dir', CANDIDATE_DIR,
        '--execute',
    ], env=provider_environment)


In [ ]:
candidate_manifest_path = CANDIDATE_DIR / 'candidate-manifest.json'
score_reports = []
if candidate_manifest_path.is_file():
    candidate_manifest = json.loads(candidate_manifest_path.read_text(encoding='utf-8'))
    assert candidate_manifest['unityInputAllowed'] is False
    candidates = [entry for entry in candidate_manifest.get('candidates', []) if entry.get('status') == 'DOWNLOADED']
    launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
    for entry in candidates:
        candidate_id = entry['candidateId']
        candidate_output = EVALUATION_DIR / candidate_id
        evaluation_report = candidate_output / 'evaluation-report.json'
        normalized_blend = candidate_output / f"{candidate_id}_normalized_NOT_PRODUCTION.blend"
        run(launcher + [
            'blender', '-b',
            '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py',
            '--',
            '--candidate', entry['modelPath'],
            '--candidate-id', candidate_id,
            '--output-dir', candidate_output,
            '--report', evaluation_report,
            '--normalized-blend', normalized_blend,
        ])
        score_report = candidate_output / 'candidate-score.json'
        run([
            sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py',
            '--reference-manifest', REFERENCE_MANIFEST,
            '--evaluation-report', evaluation_report,
            '--output', score_report,
        ])
        score_reports.append(score_report)
else:
    print('No downloaded candidates. Run the provider cell with an API key or Colab GPU.')


In [ ]:
RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
if score_reports:
    rank_command = [
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py',
        '--output', RANKING_MANIFEST,
    ]
    for score_report in score_reports:
        rank_command.extend(['--score-report', score_report])
    run(rank_command)
    ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
    assert ranking['unityInputAllowed'] is False
    if ranking.get('selectedCandidate'):
        REVIEW_DIR.mkdir(parents=True, exist_ok=True)
        launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
        run(launcher + [
            'blender', '-b',
            '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ai3d_review_asset.py',
            '--',
            '--ranking-manifest', RANKING_MANIFEST,
            '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json',
            '--output-blend', REVIEW_DIR / 'CH101_AI_AutoReview_NOT_PRODUCTION_v001.blend',
            '--report', REVIEW_DIR / 'ai3d-review-report.json',
        ])
    else:
        print('All candidates are below threshold: regeneration is required.')
else:
    print('Ranking skipped because no candidate score reports exist.')


In [ ]:
archive_base = CONTENT_ROOT / f"re-camp-{CHARACTER_CODE}-ai3d-review-NOT-PRODUCTION"
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT))
print('Archive:', archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print('Browser download is unavailable; copy the archive before the session ends.')
